# 01 · Data Download
**Ninja Van Capstone 2026 — Last-Mile Delivery Failure Prediction**

This notebook:
1. Installs all required dependencies
2. Authenticates with HuggingFace and Kaggle via Colab Secrets
3. Downloads the **LaDe** dataset (HuggingFace) and **Amazon Delivery** dataset (Kaggle)
4. Saves raw parquet snapshots to `/content/data/raw/`
5. Prints schema and row-count summaries

> **Run order:** This is Notebook 01/05 — run first.


## 1 · Install Dependencies

In [ ]:
# Install all required packages for the full pipeline
# This cell can take 2–3 minutes on first run.
import subprocess, sys

packages = [
    "datasets==2.19.0",
    "huggingface_hub==0.22.2",
    "kaggle==1.6.12",
    "pandas==2.2.2",
    "numpy==1.26.4",
    "scikit-learn==1.4.2",
    "lightgbm==4.3.0",
    "optuna==3.6.1",
    "category_encoders==2.6.3",
    "imbalanced-learn==0.12.2",
    "matplotlib==3.8.4",
    "seaborn==0.13.2",
    "missingno==0.5.2",
    "joblib==1.4.0",
    "pyarrow==16.0.0",
    "pyyaml==6.0.1",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("✅ All dependencies installed")

## 2 · Mount Repo & Set Up Paths

In [ ]:
import os, sys
from pathlib import Path

# ── Clone repo into Colab if not already present ──────────────────────────
REPO_URL  = "https://github.com/YOUR_ORG/nv-lastmile-training.git"  # update before use
REPO_DIR  = Path("/content/nv-lastmile-training")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    print(f"✅ Repo cloned → {REPO_DIR}")
else:
    print(f"ℹ️  Repo already exists at {REPO_DIR}")

# ── Add src/ to path so we can import our modules ─────────────────────────
SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Create data and output directories ────────────────────────────────────
RAW_DIR   = Path("/content/data/raw")
OUT_DIR   = Path("/content/outputs")

for d in [RAW_DIR / "lade", RAW_DIR / "amazon", OUT_DIR / "eda"]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Raw data  : {RAW_DIR}")
print(f"Outputs   : {OUT_DIR}")

## 3 · Authenticate — HuggingFace & Kaggle

> **Setup required:** In Colab, go to **Secrets** (🔑 icon in left sidebar) and add:
> - `HF_TOKEN` — your HuggingFace access token (from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens))
> - `KAGGLE_KEY` — the full JSON content of your `kaggle.json` (from [kaggle.com/account](https://www.kaggle.com/account))

In [ ]:
from google.colab import userdata

# ── HuggingFace ────────────────────────────────────────────────────────────
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✅ HuggingFace authenticated")
    else:
        print("⚠️  HF_TOKEN not set — will attempt anonymous HuggingFace download")
except Exception as e:
    HF_TOKEN = None
    print(f"⚠️  HuggingFace auth skipped: {e}")

# ── Kaggle ─────────────────────────────────────────────────────────────────
import json
try:
    KAGGLE_KEY_RAW = userdata.get("KAGGLE_KEY")
    if KAGGLE_KEY_RAW:
        kaggle_creds = json.loads(KAGGLE_KEY_RAW)
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(exist_ok=True)
        kaggle_json = kaggle_dir / "kaggle.json"
        with open(kaggle_json, "w") as f:
            json.dump(kaggle_creds, f)
        kaggle_json.chmod(0o600)
        print("✅ Kaggle credentials configured")
    else:
        kaggle_creds = None
        print("⚠️  KAGGLE_KEY not set — Amazon dataset download will be skipped")
except Exception as e:
    kaggle_creds = None
    print(f"⚠️  Kaggle auth skipped: {e}")

## 4 · Download LaDe Dataset (HuggingFace)

LaDe contains **10.67 million packages** across 6 months of real courier operations from Cainiao (Alibaba Logistics). This is the primary training dataset.

> ⏱️ Download time: ~5–10 minutes on Colab depending on connection speed.

In [ ]:
from datasets import load_dataset
import pandas as pd

print("⏳ Loading LaDe from HuggingFace...")
lade_dataset = load_dataset("Cainiao-AI/LaDe", trust_remote_code=True)

print(f"\nAvailable splits: {list(lade_dataset.keys())}")
for split_name, split_data in lade_dataset.items():
    print(f"  {split_name}: {len(split_data):,} rows, {len(split_data.features)} columns")

In [ ]:
# Save each split as parquet
lade_dfs = {}
for split_name, split_data in lade_dataset.items():
    df = split_data.to_pandas()
    df["_split"] = split_name
    save_path = RAW_DIR / "lade" / f"lade_{split_name}.parquet"
    df.to_parquet(save_path, index=False)
    lade_dfs[split_name] = df
    print(f"  Saved {split_name}: {len(df):,} rows  →  {save_path.name}")

# Combine all splits
lade_df = pd.concat(list(lade_dfs.values()), ignore_index=True)
print(f"\n✅ LaDe total: {len(lade_df):,} rows")
print(f"   Columns ({len(lade_df.columns)}): {list(lade_df.columns)}")

## 5 · Download Amazon Delivery Dataset (Kaggle)

In [ ]:
if kaggle_creds:
    import subprocess
    result = subprocess.run(
        [sys.executable, "-m", "kaggle", "datasets", "download",
         "-d", "sujalsuthar/amazon-delivery-dataset",
         "-p", str(RAW_DIR / "amazon"), "--unzip"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        # Convert CSV → parquet
        amazon_csvs = list((RAW_DIR / "amazon").glob("*.csv"))
        if amazon_csvs:
            amazon_df = pd.read_csv(amazon_csvs[0])
            amazon_df.to_parquet(RAW_DIR / "amazon" / "amazon_delivery.parquet", index=False)
            print(f"✅ Amazon Delivery: {len(amazon_df):,} rows  →  amazon_delivery.parquet")
            print(f"   Columns: {list(amazon_df.columns)}")
        else:
            print("⚠️  No CSV found after Kaggle download")
    else:
        print(f"❌ Kaggle download failed:\n{result.stderr}")
else:
    print("⏭️  Skipping Amazon dataset (no Kaggle credentials)")
    amazon_df = pd.DataFrame()
    print("   The pipeline will proceed with LaDe only.")

## 6 · Schema & Row-Count Summary

In [ ]:
from src.data_loader import print_schema_summary

print("=" * 70)
print("  DATASET INVENTORY")
print("=" * 70)

# LaDe
print(f"\n🚚 LaDe Dataset")
print(f"   Total rows: {len(lade_df):,}")
print(f"   Splits: {lade_df['_split'].value_counts().to_dict()}")
print(f"   Memory: {lade_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print_schema_summary(lade_df.head(5), "LaDe (first 5 rows shown)")

if not amazon_df.empty:
    print(f"\n📦 Amazon Delivery Dataset")
    print(f"   Total rows: {len(amazon_df):,}")
    print(f"   Memory: {amazon_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    print_schema_summary(amazon_df.head(5), "Amazon Delivery (first 5 rows shown)")

print("\n" + "=" * 70)
print("✅ NOTEBOOK 01 COMPLETE — proceed to 02_eda.ipynb")
print("=" * 70)